# 02 · Feature engineering for a fixed model (guided)

**Style A: syntax gaps.** 8 gaps (`# TODO(gN)` + `____`). The cell after each gap checks it.
Solutions: `../solutions/guided/02_feature_engineering.md`.

**Traces to** `notes/ioai-task-patterns.md` P5. In T24-O-HYPER only `LinearRegression()` with ≤ 300
features was allowed, and the author's solution was axis-swap augmentation + PCA. In T24-H-ML only a
depth-limited `DecisionTreeClassifier` was allowed. The model is frozen and the features are your only lever.

The gaps target the NumPy lines that are easy to get wrong: `axis=` arguments, reshape, fitting on train
only, and `tile` vs `repeat`.

Runtime: under 1 minute on CPU.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_friedman1, load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, accuracy_score

rmse = lambda a, b: mean_squared_error(a, b) ** 0.5

## A. A fixed `LinearRegression` on Friedman #1

$y = 10\sin(\pi x_0 x_1) + 20(x_2 - 0.5)^2 + 10x_3 + 5x_4 + \varepsilon$. Features $x_5..x_9$ are noise.
A linear model on raw inputs cannot express the sine or the square, so we hand it those terms.

In [ ]:
X, y = make_friedman1(n_samples=2000, n_features=10, noise=1.0, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0)

def fixed_linear(Ftr, Fte):             # the "frozen" model: do not change
    return rmse(yte, LinearRegression().fit(Ftr, ytr).predict(Fte))

print("raw features RMSE:", round(fixed_linear(Xtr, Xte), 3))

In [ ]:
def friedman_features(X):
    sin_term = np.sin(np.pi * X[:, 0] * X[:, 1])
    sq_term = (X[:, 2] - 0.5) ** 2
    # TODO(g1): stack the four 1-D arrays as COLUMNS -> shape (n, 4)
    return np.stack([sin_term, sq_term, X[:, 3], X[:, 4]], ____)

F = friedman_features(Xtr)
score = fixed_linear(F, friedman_features(Xte))
print(F.shape, "engineered RMSE:", round(score, 3))

In [ ]:
assert F.shape == (len(Xtr), 4), F.shape
assert score < 1.15, "should be close to the noise level (1.0)"
print("g1 ok")

## B. A fixed depth-4 decision tree on 8×8 digit images

A depth-4 tree has at most 16 leaves and splits on one feature at a time. Raw pixels are weak splitting
features; summaries of the image are strong ones.

In [ ]:
digits = load_digits()
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(digits.data, digits.target, test_size=0.3,
                                              stratify=digits.target, random_state=0)

def fixed_tree(Ftr, Fte):               # the "frozen" model: do not change
    return accuracy_score(yd_te, DecisionTreeClassifier(max_depth=4, random_state=0).fit(Ftr, yd_tr).predict(Fte))

print("raw pixels accuracy:", round(fixed_tree(Xd_tr, Xd_te), 3))

In [ ]:
# TODO(g2): turn the flat (n, 64) rows back into (n, 8, 8) images
imgs_tr = ____
imgs_te = Xd_te.reshape(-1, 8, 8)

In [ ]:
assert imgs_tr.shape == (len(Xd_tr), 8, 8)
assert np.array_equal(imgs_tr[0, 1], Xd_tr[0, 8:16]), "row 1 of image 0 should be pixels 8..15"
print("g2 ok")

In [ ]:
def shape_features(imgs):
    # TODO(g3): total ink in each ROW of each image -> shape (n, 8)
    row_mass = imgs.sum(____)
    col_mass = imgs.sum(axis=1)
    # TODO(g4): left-right asymmetry: |image - mirrored image| summed over each whole image -> shape (n,)
    lr_asym = ____
    tb_asym = np.abs(imgs - imgs[:, ::-1, :]).sum(axis=(1, 2))
    return np.column_stack([row_mass, col_mass, lr_asym, tb_asym])

S_tr, S_te = shape_features(imgs_tr), shape_features(imgs_te)
print(S_tr.shape, "accuracy with shape features:", round(fixed_tree(S_tr, S_te), 3))

In [ ]:
assert S_tr.shape == (len(imgs_tr), 18), S_tr.shape
assert np.allclose(S_tr[:, :8], [[imgs_tr[i, r, :].sum() for r in range(8)] for i in range(len(imgs_tr))])
assert np.isclose(S_tr[0, 16], np.abs(imgs_tr[0] - np.fliplr(imgs_tr[0])).sum())
print("g3, g4 ok")

PCA is fitted on the **training** images only, and the test set is projected with those components.
Refitting on test changes the axes, so the frozen model sees features with a different meaning.

Whether PCA *helps* is another question. A depth-4 tree makes 4 greedy, axis-aligned splits, and here the
shape summaries beat the PCA directions. Always measure before adding a feature block.

In [ ]:
pca = PCA(n_components=10, random_state=0).fit(Xd_tr)
Z_tr = pca.transform(Xd_tr)
# TODO(g5): project the TEST images with the PCA fitted on train (no refitting)
Z_te = ____
F_tr = np.concatenate([S_tr, Z_tr], axis=1)
F_te = np.concatenate([S_te, Z_te], axis=1)
print("PCA only:", round(fixed_tree(Z_tr, Z_te), 3), "| shape only:", round(fixed_tree(S_tr, S_te), 3),
      "| shape + PCA:", round(fixed_tree(F_tr, F_te), 3))

In [ ]:
assert np.allclose(Z_te, (Xd_te - pca.mean_) @ pca.components_.T), "Z_te must use the train-fitted PCA"
print("g5 ok")

## C. Symmetry → augmentation (the Hyperspace trick)

Here the target is invariant to any permutation of a cube's three axes. With only 150 training cubes and 216
raw features, `LinearRegression` overfits. Adding all 6 axis permutations of every training cube gives 6×
the data with the same labels.

In [ ]:
rng = np.random.default_rng(0)
w = rng.normal(size=6)
def make_cubes(n):
    Xc = rng.normal(size=(n, 6, 6, 6))
    yc = np.einsum("nijk,i,j,k->n", Xc, w, w, w) + rng.normal(0, 0.1, n)   # symmetric in (i, j, k)
    return Xc, yc
Xc_tr, yc_tr = make_cubes(150)
Xc_te, yc_te = make_cubes(1000)
flat = lambda a: a.reshape(len(a), -1)
base = rmse(yc_te, LinearRegression().fit(flat(Xc_tr), yc_tr).predict(flat(Xc_te)))
print("no augmentation RMSE:", round(base, 3))

In [ ]:
perms = list(itertools.permutations([1, 2, 3]))          # 6 orderings of the spatial axes
# TODO(g6): stack every permuted copy of the whole training set (axis 0 stays the sample axis)
X_aug = np.concatenate([____ for p in perms], axis=0)
# TODO(g7): labels for X_aug: the block order is [all samples perm0, all samples perm1, ...]
y_aug = ____
aug = rmse(yc_te, LinearRegression().fit(flat(X_aug), y_aug).predict(flat(Xc_te)))
print(X_aug.shape, "augmented RMSE:", round(aug, 3))

In [ ]:
assert X_aug.shape == (900, 6, 6, 6)
assert np.array_equal(X_aug[150], Xc_tr[0].transpose(perms[1][0] - 1, perms[1][1] - 1, perms[1][2] - 1))
assert np.array_equal(y_aug[150:300], yc_tr), "labels are misaligned (np.repeat instead of np.tile?)"
assert aug < 0.7 * base
print("g6, g7 ok")

## D. Frequency features with the FFT

Signal-like arrays (T24-H-ML's "Pulse of the Machine", `187×8`) often hide their information in frequency.
`np.fft.rfft` along the **time axis** gives the spectrum, and `np.fft.rfftfreq` gives the frequency of each bin.

In [ ]:
fs, T = 64, 256
t = np.arange(T) / fs
true_f = rng.uniform(2, 20, size=300)
sig = np.sin(2 * np.pi * true_f[:, None] * t) + 0.3 * rng.normal(size=(300, T))   # (n, T)
# TODO(g8): magnitude spectrum along the time axis -> shape (n, T//2 + 1)
spec = ____
freqs = np.fft.rfftfreq(T, d=1 / fs)
f_hat = freqs[spec.argmax(axis=1)]
print("mean |error| in Hz:", np.abs(f_hat - true_f).mean().round(3))

In [ ]:
assert spec.shape == (300, T // 2 + 1), spec.shape
assert np.abs(f_hat - true_f).mean() < 0.2
print("g8 ok, all gaps done")